# Phase 6: Phaseless DSM-DL for Inverse Scattering

**Project**: Demystifying Iterative Direct Sampling Methods — From Theory to Code  
**Reference**: Ning, Han, and Zou, *A Direct Sampling Method and Its Integration with Deep Learning for Inverse Scattering Problems with Phaseless Data* (arXiv:2403.02584, v2)  
**Objective**: Build a tutorial-style Python reproduction of Section 5, while also porting the provided MATLAB/PyTorch reference pipeline into clean Python modules.

---

Phase 6 extends Phases 1–5 from elliptic/parabolic IDSM to **phaseless inverse scattering**. On
$$
\Omega=[-1,1]^2,
$$
we use the paper models:

- **impenetrable scatterers**
$$
\Delta u + k^2u=0\ \text{in }\mathbb R^2\setminus D,\qquad
u|_{\partial D}=0\ (\text{soft})\ \text{or}\ \partial_\nu u|_{\partial D}=0\ (\text{hard}),
$$
- **inhomogeneous medium**
$$
\Delta u + k^2 n(x)u=0\ \text{in }\mathbb R^2\quad (\text{Eq. 2.4}).
$$

We only observe noisy magnitudes on a receiver circle $\Gamma_r$:
$$
|u_\delta(x_r)|=|u(x_r)|+\delta\,\zeta(x_r)\,\|u\|_2\quad (\text{Eq. 5.1}).
$$
The phaseless DSM index uses
$$
\Delta(x_r,d)=\frac{|u|^2-|u^i|^2}{u^i}\quad (\text{Eq. 3.12}),\qquad
I^{\text{phaseless}}_{\text{DSM}}(z)=\left|\int_{\Gamma_r}G(z,x_r)\Delta(x_r,d)\,ds\right|\quad (\text{Eq. 3.11}).
$$

DSM-DL then learns the map
$$
\{I_{\text{DSM},i}\}_{i=1}^{N_i}\longrightarrow n(x)
$$
with U-Net and Eq. (4.1):
$$
\mathcal L=\|X-Y\|_2^2+\alpha_1\,\text{TV}(X)+\alpha_2(1-\text{SSIM}(X,Y)),\qquad \alpha_1=\alpha_2=0.5.
$$

This notebook keeps the same teaching style as 01–05:

1. theory block with equation-to-code mapping,
2. runnable DSM experiments (Section 5.1),
3. paper-scale outputs loaded from scripts,
4. concise summary of what is reproduced.

### Two reproducible Python paths in this repository

- **Paper path**: `scripts/run_phaseless_full_repro.py` + `src/phaseless_scattering.py` + `src/phaseless_dsmdl.py`.
- **Reference-code path**: `scripts/run_phaseless_forward_generate.py` + `scripts/run_phaseless_port_repro.py` + `src/phaseless_reference.py`.

Both are pure Python. The paper path is used for Fig.2–10 and Table 1–2 comparison.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import torch

sys.path.insert(0, os.path.abspath('..'))

from src.plot_style import apply_idsm_plot_style, save_figure, contrast_norm
from src.phaseless_scattering import (
    PhaselessDSMConfig,
    example_specs,
    run_example_dsm_paper,
    topk_peak_hit_rate,
    peak_location_error,
)
from src.phaseless_dsmdl import DatasetConfig  # used downstream when describing strict-protocol settings

apply_idsm_plot_style()
FIG_DIR = Path('../figures/06_phaseless')
FIG_DIR.mkdir(parents=True, exist_ok=True)

print('torch:', torch.__version__)
print('figure dir:', FIG_DIR.resolve())

## 1. Phaseless DSM setup (paper Section 5.1)

We use the paper defaults in live cells:

- wavelength $\lambda=0.75$, $k=2\pi/\lambda$,
- sampling domain $\Omega=[-1,1]^2$,
- one or multiple plane-wave incidences,
- 100 receivers on radius-4 circle,
- noisy phaseless data from Eq. (5.1),
- corrected data $\Delta$ from Eq. (3.12), indicator from Eq. (3.11).

Section 5.2 training/evaluation outputs are loaded from scripts:

- `scripts/run_phaseless_full_repro.py` (paper protocol),
- `scripts/run_phaseless_port_repro.py` (reference DSMDL path, Python port).

So notebook cells stay lightweight and tutorial-focused, while full training is reproducible from command line.

In [ ]:
from src.phaseless_scattering import run_example_dsm_paper

dsm_scan_grid = 160  # paper-style high-resolution scan grid

def _show_truth(ax, mask, kind):
    if kind == 'medium':
        field = 1.0 + 2.0 * mask.astype(np.float32)
        im = ax.imshow(field, origin='lower', extent=(-1, 1, -1, 1), cmap='viridis', vmin=1.0, vmax=3.0)
    else:
        im = ax.imshow(1.0 - mask.astype(np.float32), origin='lower', extent=(-1, 1, -1, 1), cmap='viridis', vmin=0.0, vmax=1.0)
    plt.colorbar(im, ax=ax, shrink=0.8)

def _show_indicator(ax, indicator, mask):
    overlay = indicator.copy()
    overlay[mask.astype(bool)] = 0.0
    im = ax.imshow(overlay, origin='lower', extent=(-1, 1, -1, 1), cmap='viridis', vmin=0.0, vmax=1.0)
    plt.colorbar(im, ax=ax, shrink=0.8)


## 2. Example 1--3: one incidence, noise robustness

Paper Figure 2--4 compares ground truth with reconstructions at $\delta=5\%$ and $10\%$.
We run the same pattern for:

- Example 1: one medium square of width 0.15 at the origin with $n(x)=3$,
- Example 2: two well-separated sound-soft squares of width 0.15,
- Example 3: two close medium squares of width 0.15 with $n(x)=3$.

In [ ]:
trip = [
    ('ex1_medium_square', 'medium'),
    ('ex2_sound_soft_squares', 'obstacle'),
    ('ex3_close_medium_squares', 'medium'),
]
results = {}
for key, _ in trip:
    results[(key, 0.05, 1)] = run_example_dsm_paper(example_key=key, noise_level=0.05, n_incident=1, seed=0, scan_grid_size=dsm_scan_grid)
    results[(key, 0.10, 1)] = run_example_dsm_paper(example_key=key, noise_level=0.10, n_incident=1, seed=0, scan_grid_size=dsm_scan_grid)

fig, axes = plt.subplots(3, 3, figsize=(12.5, 9.0))
for i, (key, kind) in enumerate(trip):
    out5 = results[(key, 0.05, 1)]
    out10 = results[(key, 0.10, 1)]
    _show_truth(axes[i, 0], out5['truth_mask'], kind)
    _show_indicator(axes[i, 1], out5['indicator'], out5['truth_mask'])
    _show_indicator(axes[i, 2], out10['indicator'], out10['truth_mask'])
    axes[i, 0].set_title(f'Ex{i+1} (a) truth', fontsize=9)
    axes[i, 1].set_title(f'Ex{i+1} (b) $\\delta=5\\%$', fontsize=9)
    axes[i, 2].set_title(f'Ex{i+1} (c) $\\delta=10\\%$', fontsize=9)
    for j in range(3):
        axes[i, j].set_xticks([-1, 0, 1])
        axes[i, j].set_yticks([-1, 0, 1])

fig.suptitle('Phaseless DSM — Examples 1-3 (paper Fig.2-4)', fontsize=11)
fig.tight_layout(rect=[0, 0, 1, 0.97])
save_figure(fig, FIG_DIR / '06_fig2_4_examples1_3.png', dpi=170)
plt.show()

## 3. Example 4: ring with $N_i=1,3,5$ (paper Figure 5 trend)

Following Eq. (5.2), we average indicators over multiple incidences and check whether ring reconstruction quality improves as $N_i$ increases.

In [ ]:
ni_list = [1, 3, 5]
for ni in ni_list:
    results[('ex4_medium_ring', 0.05, ni)] = run_example_dsm_paper(
        example_key='ex4_medium_ring', noise_level=0.05, n_incident=ni, seed=0, scan_grid_size=dsm_scan_grid
    )
    results[('ex4_medium_ring', 0.10, ni)] = run_example_dsm_paper(
        example_key='ex4_medium_ring', noise_level=0.10, n_incident=ni, seed=0, scan_grid_size=dsm_scan_grid
    )

fig, axes = plt.subplots(3, 3, figsize=(12.5, 9.0))
for row, ni in enumerate(ni_list):
    out5 = results[('ex4_medium_ring', 0.05, ni)]
    out10 = results[('ex4_medium_ring', 0.10, ni)]
    _show_truth(axes[row, 0], out5['truth_mask'], 'medium')
    _show_indicator(axes[row, 1], out5['indicator'], out5['truth_mask'])
    _show_indicator(axes[row, 2], out10['indicator'], out10['truth_mask'])
    axes[row, 0].set_title(f'$N_i={ni}$ (a) truth', fontsize=9)
    axes[row, 1].set_title(f'$N_i={ni}$ (b) $\\delta=5\\%$', fontsize=9)
    axes[row, 2].set_title(f'$N_i={ni}$ (c) $\\delta=10\\%$', fontsize=9)
    for j in range(3):
        axes[row, j].set_xticks([-1, 0, 1])
        axes[row, j].set_yticks([-1, 0, 1])

fig.suptitle('Phaseless DSM — Example 4 ring (paper Fig.5)', fontsize=11)
fig.tight_layout(rect=[0, 0, 1, 0.97])
save_figure(fig, FIG_DIR / '06_fig5_ring_ni_sweep.png', dpi=170)
plt.show()

## 4. DSM quantitative sanity checks

For each run we report:

- top-k peak hit rate (support overlap of strongest indicator pixels),
- peak location error (distance between max-indicator point and truth centroid).

These numbers complement the visual checks and are used to verify paper-consistent trends.

In [ ]:
rows = []
for (key, noise, ni), out in results.items():
    hit = topk_peak_hit_rate(out['indicator'], out['truth_mask'], k_fraction=0.02)
    err = peak_location_error(out['indicator'], out['truth_mask'], out['grid_x'], out['grid_y'])
    rows.append((key, ni, noise, hit, err))

rows = sorted(rows, key=lambda x: (x[0], x[1], x[2]))
print(f"{'example':30s} {'Ni':>3s} {'noise':>7s} {'hit@2%':>9s} {'peak_err':>10s}")
for r in rows:
    print(f"{r[0]:30s} {r[1]:3d} {r[2]:7.2f} {r[3]:9.3f} {r[4]:10.4f}")

## 5. Full training outputs (loaded from script artifacts)

Run commands (outside notebook):

```bash
conda run -n IDSM python scripts/run_phaseless_full_repro.py --mnist-download
conda run -n IDSM python scripts/run_phaseless_port_repro.py
```

Produced artifacts:

- `results/phaseless/full_summary.json`, `results/phaseless/full_comparison.md` (paper path),
- `results/phaseless/reference_summary.json`, `results/phaseless/reference_comparison.md` (reference-code path),
- `results/phaseless/checkpoints/*.pt`, `figures/06_phaseless/*.png`.

The next cell loads these files and prints concise numerical comparisons.

In [ ]:
import json

results_dir = Path('../results/phaseless')
summary_path = results_dir / 'full_summary.json'
reference_summary_path = results_dir / 'reference_summary.json'

paper_table1 = {
    'Ni=1,delta=0.02': 0.9949, 'Ni=1,delta=0.10': 0.9772,
    'Ni=4,delta=0.02': 0.9977, 'Ni=4,delta=0.10': 0.9916,
}
paper_table2 = {
    'mnist,Ni=4,delta=0.05': 0.0827, 'mnist,Ni=4,delta=0.10': 0.1043,
    'mnist,Ni=16,delta=0.05': 0.0617, 'mnist,Ni=16,delta=0.10': 0.0755,
    'chinese_like,Ni=4,delta=0.05': 0.1096, 'chinese_like,Ni=4,delta=0.10': 0.1252,
    'chinese_like,Ni=16,delta=0.05': 0.0721, 'chinese_like,Ni=16,delta=0.10': 0.0854,
    'austria_ring_1,Ni=4,delta=0.05': 0.1163, 'austria_ring_1,Ni=4,delta=0.10': 0.1258,
    'austria_ring_1,Ni=16,delta=0.05': 0.0851, 'austria_ring_1,Ni=16,delta=0.10': 0.0922,
    'austria_ring_2,Ni=4,delta=0.05': 0.1897, 'austria_ring_2,Ni=4,delta=0.10': 0.1810,
    'austria_ring_2,Ni=16,delta=0.05': 0.1260, 'austria_ring_2,Ni=16,delta=0.10': 0.1367,
}

if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print(f"paper run timestamp: {summary['run_at']}  device: {summary['device']}")
    poly = summary['dsmdl']['polygon']['accuracy']
    mnist = summary['dsmdl']['mnist_family']['relative_l2']

    print('\nTable 1  Polygon accuracy (paper Section 5.2.1.1)')
    print(f"{'case':<22s}{'paper':>10s}{'ours':>10s}{'diff':>10s}")
    for k, v in paper_table1.items():
        ours = poly.get(k, float('nan'))
        print(f"{k:<22s}{v:>10.4f}{ours:>10.4f}{ours - v:>+10.4f}")

    print('\nTable 2  Relative L2 (paper Section 5.2.2)')
    print(f"{'case':<35s}{'paper':>10s}{'ours':>10s}{'diff':>10s}")
    for k, v in paper_table2.items():
        ours = mnist.get(k, float('nan'))
        print(f"{k:<35s}{v:>10.4f}{ours:>10.4f}{ours - v:>+10.4f}")

    print('\nSection 5.2.3  mixed-circle (phased training, phaseless test)')
    for k, v in summary['dsmdl']['mixed_circle']['metrics'].items():
        print(f"  {k:<32s}{v:.4f}")
else:
    print('Paper-scale summary missing: run scripts/run_phaseless_full_repro.py first.')

if reference_summary_path.exists():
    reference_summary = json.loads(reference_summary_path.read_text())
    print('\nReference DSMDL Python port summary')
    print(f"  n_incident: {reference_summary['n_incident']}")
    print(f"  best_test_rel_l2: {reference_summary['best_test_rel_l2']:.6f}")
    print(f"  final_test_rel_l2: {reference_summary['final_test_rel_l2']:.6f}")
else:
    print('Reference summary missing: run scripts/run_phaseless_port_repro.py first.')

## 6. DSM-DL figure gallery (paper Fig.6--10)

The full script writes the DSM-DL reconstruction panels as image artifacts. We load them here so the notebook itself displays every paper-scale visual result: polygon (Fig.6), MNIST (Fig.7), Chinese-character OOD (Fig.8), Austria-ring OOD (Fig.9), and mixed-circle phased-training / phaseless-testing (Fig.10).

In [ ]:
from IPython.display import Image, display

figure_paths = [
    FIG_DIR / 'fig6_polygon_recon.png',
    FIG_DIR / 'fig7_mnist_recon.png',
    FIG_DIR / 'fig8_chinese_recon.png',
    FIG_DIR / 'fig9_austria_recon.png',
    FIG_DIR / 'fig10_mixed_recon.png',
]
for path in figure_paths:
    if path.exists():
        print(path.name)
        display(Image(filename=str(path)))
    else:
        print(f'missing: {path} (run scripts/run_phaseless_full_repro.py first)')

## 7. Summary and discussion

### Phase 6 deliverables

| Paper component | Notebook reference |
|---|---|
| Helmholtz forward + Eq. (3.11)/(3.12) phaseless DSM | Sections 1-3 call `run_example_dsm_paper` (VIE for Ex 1/3/4 medium scatterers, BIE Dirichlet for Ex 2 sound-soft squares) with paper-style viridis overlays |
| Section 5.1 Examples 1–4 figures | Sections 2–3 (`06_fig2_4_examples1_3.png`, `06_fig5_ring_ni_sweep.png`) |
| DSM peak / support diagnostics | Section 4 |
| Section 5.2.1 polygon DSM-DL (Table 1) | Section 5 comparison cell + Section 6 `fig6_polygon_recon.png` |
| Section 5.2.2 MNIST + Chinese / Austria OOD (Table 2) | Section 5 comparison cell + Section 6 `fig7_mnist_recon.png` / `fig8_chinese_recon.png` / `fig9_austria_recon.png` |
| Section 5.2.3 mixed-circle (phased train → phaseless test) | Section 5 metrics block + Section 6 `fig10_mixed_recon.png` |
| Paper Table 1 / Table 2 comparison | Section 5 |

### Reference-code Python port

The original reference packages are now runnable in Python-only form:

- forward generation (MATLAB `DataMnist.m` logic): `scripts/run_phaseless_forward_generate.py`,
- DSMDL training (PyTorch `main.py` + `U_Net3Ab` logic): `scripts/run_phaseless_port_repro.py`,
- API module: `src/phaseless_reference.py`.

This path is kept separate from the strict paper path so both can be reproduced and compared clearly.

### Key claim of Section 5.2.3 (phased-trained → phaseless inference)

The paper's headline observation is that the DSM-DL network trained with **phased** index functions (Eq. 3.1) can still produce accurate reconstructions when the runtime evaluation receives only **phaseless** index functions (Eq. 3.11). The script `scripts/run_phaseless_full_repro.py` honors this by passing `phased_training=True` to the mixed-circle training case and then evaluating it with the phaseless simulator; the numerical evidence is loaded in Sections 5--6 above.

### Comparison with previous phases

| Aspect | Phases 1–5 (elliptic/parabolic) | Phase 6 (Helmholtz phaseless) |
|---|---|---|
| Forward operator | $-\nabla\!\cdot(\sigma\nabla\cdot)$ (+ time / potential) | $\Delta + k^2 n(x)$ |
| Measurement | Cauchy pair $(f, y^d)$ | Phaseless magnitude $|u|$ on $\Gamma_r$ |
| Inversion path | IDSM / DSM with regularized DtN | DSM index → U-Net regression |
| Loss / regularization | Implicit via low-rank DFP/BFG | Explicit Eq. (4.1) with TV + SSIM |
| Coefficient recovery | $\sigma$, $V$, $U$ time-dependent | $n(x)$ with cutoff $\{0,1,3\}$ |

The essential new ingredient is the dual use of DSM as a robust prior for a learned inverse, which lets a single network handle medium scatterers and impenetrable scatterers in a unified framework even when only phaseless data is available.